# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

S. Torniamenti  ->  S. Torniamenti  |  ['S. Torniamenti']
S. Jiao  ->  S. Jiao  |  ['S. Jiao']
H. Beuther  ->  H. Beuther  |  ['H. Beuther']
K. Doi  ->  K. Doi  |  ['K. Doi']
J. d. Brok  ->  J. d. Brok  |  ['J. d. Brok']
F. Xu  ->  F. Xu  |  ['F. Xu']
S. D. Reyes-Reyes  ->  S. D. Reyes-Reyes  |  ['S. D. Reyes-Reyes']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
J. Li  ->  J. Li  |  ['J. Li']
T. Henning  ->  T. Henning  |  ['T. Henning']
Y. Wang  ->  Y. Wang  |  ['Y. Wang']
Arxiv has 89 new papers today
          6 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/6 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.03812
extracting tarball to tmp_2611.03812...

 done.


S. Torniamenti  ->  S. Torniamenti  |  ['S. Torniamenti']


Retrieving document from  https://arxiv.org/e-print/2611.04319
extracting tarball to tmp_2611.04319...

 done.


... (3 outputs left out)

 done.
Retrieving document from  https://arxiv.org/e-print/2611.04976
extracting tarball to tmp_2611.04976... done.


T. Henning  ->  T. Henning  |  ['T. Henning']


Found 97 bibliographic references in tmp_2611.04976/aa60918-26.bbl.
Retrieving document from  https://arxiv.org/e-print/2611.05253
extracting tarball to tmp_2611.05253...

 done.


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.03812-b31b1b.svg)](https://arxiv.org/abs/2611.03812) | **The STARfall pathway: forging intermediate-mass black holes through stellar collisions in dense star clusters**  |
|| F. Simonato, et al. -- incl., <mark>S. Torniamenti</mark> |
|*Appeared on*| *2026-11-04*|
|*Comments*| *22 pages, 15 figures, 3 tables*|
|**Abstract**|            Intermediate-mass black holes (IMBHs, $10^2 < m_{\rm BH}/\mathrm{M}_{\odot} < 10^5$) are elusive objects bridging the gap between stellar-mass and supermassive black holes, whose origin remains uncertain. They may form from the collapse of very massive stars (VMSs, $m_*\geq150\,\mathrm{M}_{\odot}$) assembled through repeated stellar collisions in compact and fast evolving star clusters. We investigate the formation of VMSs and IMBH seeds via repeated mergers and collisions as a function of cluster initial conditions, assessing the implications for globular clusters and dense high-redshift system. We develop STARfall, a semi-analytic tool that follows the evolution of a massive stellar binary via binary-single interactions, mergers, collisions, and stellar evolution, while consistently accounting for the host cluster evolution. Our results show that clusters with central densities above $10^6\,\mathrm{M}_{\odot}\,\mathrm{pc}^{-3}$ can undergo a catastrophic sequence of stellar collisions, or "starastrophe", within $3$ Myr. Through this process, VMSs rapidly grow in mass and collapse into IMBHs with masses of up to $10^3-10^4\,\mathrm{M}_{\odot}$, despite the effects of intense stellar winds. Applying our models to observed systems, we find that compact clusters in the Cosmic Gems Arc and Milky Way globular clusters could have hosted VMSs that collapsed into IMBHs in this mass range. Despite their longer core-collapse timescales, the inferred central densities of Little Red Dots may also support IMBH formation in the same mass interval. Our models show that VMS assembly requires two essential ingredients: short core-collapse times and large central densities. Compared to observed systems, our models suggest that clusters in the Cosmic Gems Arcs and several Galactic globular clusters could have nurtured VMSs that directly collapsed to IMBHs with masses $10^3-10^4\,\text{M}_{\odot}$.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.04319-b31b1b.svg)](https://arxiv.org/abs/2611.04319) | **Constraining Dust Properties Along the Minor Axis of Andromeda**  |
|| <mark>S. Jiao</mark>, et al. -- incl., <mark>H. Beuther</mark>, <mark>K. Doi</mark>, <mark>J. d. Brok</mark>, <mark>F. Xu</mark>, <mark>S. D. Reyes-Reyes</mark> |
|*Appeared on*| *2026-11-04*|
|*Comments*| *15 pages, 12 figures. Accepted for publication in A&A*|
|**Abstract**|            Previous studies of dust emission in M31 based primarily on Herschel data have reported inconsistent radial trends in the dust emissivity index ($\beta$), partly due to limited wavelength coverage and the intrinsic $T_{\rm dust}$-$\beta$ degeneracy. We aim to improve the constraints of $\beta$ determinations and obtain the spatial surface density and temperature distributions at GMC scales in M31 by extending the spectral coverage into the millimeter regime and by quantifying the impact of wavelength coverage on the $T_{\rm dust}$-$\beta$ degeneracy. We present new IRAM 30m NIKA2 1.15 mm observations along the minor axis of M31 at an angular resolution of 11 arcsec, corresponding to $\sim 40$ pc at the distance of M31. These data are used with Herschel and JCMT SCUBA-2 maps to perform pixel-by-pixel single-component modified blackbody fitting. Including the 850 $\mu$m and 1.15 mm data helps reduce the $T_{\rm dust}$-$\beta$ degeneracy and yields an average emissivity index of $\beta \approx 1.8$, systematically lower than values derived from far-infrared-only analyses. Radial analysis shows higher $\beta$ values in the 5 kpc ring compared to the 10 kpc ring. The elevated $\beta$ values in the inner 5 kpc may be linked to changes in dust composition, potentially associated with enhanced starlight heating in this region. Millimeter wavelength coverage is important for improving $\beta$ constraints in M31. The lower $\beta$ values inferred from the 250--1150 $\mu$m SED fitting suggest possible variations in dust emissivity and highlight the importance of sampling the long-wavelength SED slope in nearby galaxies.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.04976-b31b1b.svg)](https://arxiv.org/abs/2611.04976) | **The ALMA survey to Resolve exoKuiper belt Substructures (ARKS) XII. Constraints on large bodies embedded in discs**  |
|| M. R. Jankovic, et al. -- incl., <mark>T. Henning</mark> |
|*Appeared on*| *2026-11-04*|
|*Comments*| *accepted for publication in A&A*|
|**Abstract**|            The ALMA survey to Resolve exoKuiper belt Substructures (ARKS) presents detailed constraints on structure and properties of a sample of 24 debris discs at millimetre wavelengths, including measurements of vertical thicknesses for a subsample of discs. We estimated sizes and total masses of large bodies that feed collisional cascades producing the observed millimetre dust in 11 ARKS discs with a well constrained vertical thickness. Separately, we estimated the sizes and total masses of large bodies that could be gravitationally stirring these discs, assuming that such stirring is the origin of the vertical thickness. We used a semi-analytic particle-in-a-box collisional model and a semi-analytic model of stirring through gravitational scattering. We accounted for the dependence of collisional damping efficiency on the average collision velocity. We find that planetesimals feeding the collisional cascades are less than 20 km in size, while the total masses in the planetesimals are lower than 200 M$_\oplus$. There is a good agreement between minimum disc masses inferred from millimetre and infrared observations. The hypothetical stirrers are always larger in size than the planetesimals, but they do not necessarily contain most of the disc mass. In agreement with previous work, the observed dust masses can be explained by steady-state collisional models with reasonable disc masses. The measured vertical scale-heights can be explained by gravitational stirring from embedded large bodies of reasonable properties. However, more advanced models are needed to address the observed non-Gaussian vertical profiles, along with the dependence of disc scale-height on particle size and stellocentric distance.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.04544-b31b1b.svg)](https://arxiv.org/abs/2611.04544) | **The Survival Paradox of Close-In Planets Orbiting Red Clump Giants**  |
|| W.-X. Lin, et al. -- incl., <mark>H. Zhang</mark> |
|*Appeared on*| *2026-11-04*|
|*Comments*| *10 pages, 3 figures, 1 table*|
|**Abstract**|            Close-in planets orbiting core-helium-burning red-clump (RC) giants directly challenge tidal-engulfment theory, which predicts that planets within about 1 au of Sun-like stars should be engulfed during red-giant-branch (RGB) expansion. The existence of such survivors would imply missing physical processes in the late stages of stellar and planetary evolution. Yet only one such "forbidden" planet had been identified to date. Here we report three systems-24 Boo b, HD 12648 b, and TYC 4282-605-1 b-whose host stars are confirmed red-clump giants through asteroseismology, showing that each has already undergone dramatic expansion and contraction following helium ignition. Their planets now reside at orbital separations smaller than the maximum stellar radii attained during the RGB phase, implying that they should not have survived. Among them, 24 Boo b orbits at only 0.183 au with a 30-day period, making it the closest planet known around a red-clump giant. These discoveries demonstrate that this paradox is not an isolated anomaly but signals the emergence of a new class of evolved planetary systems. Their survival defies current models of stellar-planetary coevolution and points to unexplored physics shaping planetary architectures around post-RGB stars.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.04605-b31b1b.svg)](https://arxiv.org/abs/2611.04605) | **Probing the Faint Outskirts of Star-Forming Galaxies with 1<z<9 Using Stacked Images**  |
|| C. Jia, et al. -- incl., <mark>H. Zhang</mark>, <mark>J. Li</mark> |
|*Appeared on*| *2026-11-04*|
|*Comments*| *19 pages, 15 figures, accepted for publication in ApJL*|
|**Abstract**|            We present a systematic study of the faint outer light profiles of star-forming galaxies using deep JWST/NIRCam imaging from the CEERS, JADES, and PRIMER surveys. Our sample consists of 14,812 face-on disk galaxies at $1<z<9$ with $\log(M_*/M_{\odot})>8.5$, selected from the DAWN JWST Archive. To characterize the low-surface-brightness outskirts beyond the effective radius ($R_e$), we stack galaxies in bins of stellar mass and redshift, accounting for cosmological surface-brightness dimming, angular-size scaling, and PSF convolution, and estimate residual backgrounds from blank-sky stacks. The color profiles generally become bluer toward larger radii, indicating negative color gradients and supporting an inside-out growth scenario. Galaxies with $\log(M_*/M_{\odot}) < 9.5$ at $z>2.5$ exhibit a weak reversal in their outermost regions, suggesting more complex stellar population gradients at early times. The outer light profiles generally exhibit a good exponential decline with radius, and the outer scale lengths measured beyond $2R_e$ show mass and redshift dependences remarkably similar to those of the conventional mass-size relation. This similarity suggests that the extended outer light is closely linked to the same processes governing the growth of the inner galaxy, rather than providing a direct tracer of ex-situ stellar components in the early universe. We also find little wavelength dependence in the outer scale lengths, indicating that dust attenuation likely has a limited impact at these radii. These results provide new constraints on the buildup of extended stellar distributions during cosmic noon and at earlier epochs.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.05253-b31b1b.svg)](https://arxiv.org/abs/2611.05253) | **CCAT: Prime-Cam Control System and Related Software**  |
|| A. Middleton, et al. -- incl., <mark>Y. Wang</mark> |
|*Appeared on*| *2026-11-04*|
|*Comments*| *11 pages, 8 figures, as submitted to SPIE Astronomical Telescopes + Instrumentation*|
|**Abstract**|            Prime-Cam is one of the primary instruments being developed for use by the CCAT collaboration with the Fred Young Submillimeter Telescope (FYST), which is undergoing construction at a height of 5600 m in Cerro Chajnantor. Prime-Cam can host up to seven instrument modules with kinetic inductance detectors (KIDs), including broadband modules ranging 280-850 GHz and imaging spectrometers for line-intensity mapping. Prime-Cam requires a number of hardware devices for standard operation and data collection, and so a distributed control system must be developed for software integration of these devices and user-friendliness when controlling Prime-Cam in the field, especially during remote operation that will be typical during observations. The Prime-Cam Control System (PCS) is being developed using the Observatory Control System (OCS) framework originally developed for Simons Observatory (SO) and is adapted for Prime-Cam and CCAT-specific requirements. Within OCS/PCS, hardware devices are controlled individually by long-running software processes known as agents, which are typically deployed in Docker containers running in the background of one of the Prime-Cam network computers. These agents are controlled by client scripts through the command line or frontend tools like Nextline and PCS-web. PCS also makes use of several external open source software tools for remote operations, data management, and logging. This system is being tested in the lab in preparation for deployment in 2026 for use with Prime-Cam on FYST.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.03812/./Figure/histo_m1_MCLU6_new.png', 'tmp_2611.03812/./Figure/plot_m1_vs_time_Z0.0002_Mcl6_winds_coll.png', 'tmp_2611.03812/./Figure/contour_coll_vms.png']
copying  tmp_2611.03812/./Figure/histo_m1_MCLU6_new.png to _build/html/
copying  tmp_2611.03812/./Figure/plot_m1_vs_time_Z0.0002_Mcl6_winds_coll.png to _build/html/
copying  tmp_2611.03812/./Figure/contour_coll_vms.png to _build/html/
exported in  _build/html/2611.03812.md
    + _build/html/tmp_2611.03812/./Figure/histo_m1_MCLU6_new.png
    + _build/html/tmp_2611.03812/./Figure/plot_m1_vs_time_Z0.0002_Mcl6_winds_coll.png
    + _build/html/tmp_2611.03812/./Figure/contour_coll_vms.png
found figures ['tmp_2611.04319/./Figures/Figure3.png', 'tmp_2611.04319/./Figures/Figure4_radial_distribution_Sigma_T_beta.png', 'tmp_2611.04319/./Figures/Figure6.png']
copying  tmp_2611.04319/./Figures/Figure3.png to _build/html/
copying  tmp_2611.04319/./Figures/Figure4_radial_distribution_Sigma_T_beta.png to _build/html/
cop

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\Ms}{{\rm M}_\odot}$
$\newcommand{\kms}{{\rm ~km}~{\rm s}^{-1}}$
$\newcommand{\df}{{\rm df}}$
$\newcommand{\bh}{{\rm BH}}$
$\newcommand{\ns}{{\rm NS}}$
$\newcommand{\co}{{\rm CO}}$
$\newcommand{\gws}{{\rm GW190814}}$
$\newcommand{\yrgpc}{{\rm ~yr}^{-1}{\rm ~Gpc}^{-3}}$
$\newcommand{\lvk}{{\rm LVC}}$
$\newcommand{\gw}{{\rm GW}}$
$\newcommand{\mer}{{\rm mer}}$
$\newcommand{\der}{{\rm d}}$
$\newcommand{\smbh}{{\rm SMBH}}$
$\newcommand{\imbh}{{\rm IMBH}}$
$\newcommand{\ARGdf}{\texttt{ARGdf}}$
$\newcommand{\archain}{\texttt{ARCHAIN}}$
$\newcommand{\himri}{{\rm HIMRI}}$
$\newcommand{\hor}{{\rm hor}}$
$\newcommand{\snr}{{\rm S/N}}$
$\newcommand{\obs}{{\rm obs}}$
$\newcommand{\gc}{{\rm GC}}$
$\newcommand{\Log}{{\rm Log}}$
$\newcommand{\inn}{{\rm in}}$
$\newcommand{\out}{{\rm out}}$
$\newcommand{\as}[1]{\textcolor{red}{\it{\textbf{[AS: #1]}}} }$
$\newcommand{\fil}[1]{\textcolor{green}{[Filippo: #1]}}$
$\newcommand{\sevn}{\textsc{sevn}}$
$\newcommand{\starfall}{\texttt{STARfall}}$
$\newcommand{\ham}{{\rm HM}}$
$\newcommand{\dragon}{\textsc{Dragon-II}}$
$\newcommand{\bpop}{\texttt{B-POP}}$
$\newcommand{\arraystretch}{1.6}$
$\newcommand{\arraystretch}{1.0}$
$\newcommand{\arraystretch}{1.9}$
$\newcommand{\arraystretch}{1.9}$</div>



<div id="title">

# The STARfall pathway: forging intermediate-mass black holes through stellar collisions in dense star clusters

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.03812-b31b1b.svg)](https://arxiv.org/abs/2611.03812)<mark>Appeared on: 2026-11-04</mark> -  _22 pages, 15 figures, 3 tables_

</div>
<div id="authors">

F. Simonato, et al. -- incl., <mark>S. Torniamenti</mark>

</div>
<div id="abstract">

**Abstract:** Intermediate-mass black holes (IMBHs, $10^2 < m_{\rm BH}/\text{M}_{\odot} < 10^5$ ) are elusive objects falling between stellar-mass and supermassive black holes, whose origin is still highly uncertain. These objects may possibly form from the collapse of very massive stars (VMSs, $m_*\geq 150 \Ms$ ) assembled through stellar collisions in compact and fast evolving star clusters. We investigate the formation of VMSs and IMBH seeds via repeated mergers and collisions as a function of cluster initial conditions, assessing the implications for globular clusters and dense high-redshift systems. We develop $\starfall$ , a semi-analytic tool that follows the evolution of a massive stellar binary via binary-single interactions, mergers, collisions, and stellar evolution, while consistently accounting for the host cluster evolution. Our results suggest that clusters with central densities $> 10^6 \Ms \text{pc}^{-3}$ undergo a catastrophic series of stellar collisions, or "starastrophe", on timescales $< 3$ Myr. Through this process, VMSs rapidly grow in mass and eventually collapse into IMBHs with masses of up to $10^3$ – $10^4 \Ms$ , despite the effects of intense stellar winds. Applying our results to the compact stellar clusters observed in the Cosmic Gems Arc, we find that these environments could promote the formation of IMBHs within this mass range. Similarly, our models suggest that some Milky Way globular clusters may have hosted IMBH seeds with comparable natal masses. Finally, applying our framework to Little Red Dots, we find that, despite their longer core-collapse timescales, their inferred core densities can still support the formation of IMBHs in the same mass interval. Our models show that VMS assembly requires two essential ingredients: short core-collapse times and large central densities. In the explored density range, $> 10^6 \Ms \text{pc}^{-3}$ , VMSs attain masses up to $10^3-10^5 \Ms$ . Compared to observed systems, our model suggest that clusters in the Cosmic Gems Arcs and several Galactic globular clusters could have nurtured VMSs that directly collapsed to IMBHs with masses $10^3-10^4 \Ms$ .

</div>

<div id="div_fig1">

<img src="tmp_2611.03812/./Figure/histo_m1_MCLU6_new.png" alt="Fig9" width="100%"/>

**Figure 9. -** Stellar mass distribution (upper panels) and IMBH mass distribution (lower panels) for clusters with mass $M_{\rm cl,0} = 10^6 $\Ms$$ and half-mass radius $R_{\rm hm,0}$ in the range $0.05$–$1$ pc. The left (right) panels correspond to metallicity $Z = 0.0002$($Z = 0.002$). (*fig:histo_vms_imbh*)

</div>
<div id="div_fig2">

<img src="tmp_2611.03812/./Figure/plot_m1_vs_time_Z0.0002_Mcl6_winds_coll.png" alt="Fig8" width="100%"/>

**Figure 8. -** Mass growth of VMSs (colored lines) in clusters of $M_{\rm cl,0} = 10^6  $\Ms$$ and $R_{\rm hm,0} = 0.05, 0.1,0.5,1$ pc. Black lines represent the evolution of the cluster core radius. The vertical gray dashed lines mark the beginning of the core-collapse phase. (*fig:vms_mass_growth*)

</div>
<div id="div_fig3">

<img src="tmp_2611.03812/./Figure/contour_coll_vms.png" alt="Fig2" width="100%"/>

**Figure 2. -** Contour plots in the $M_{\rm cl}$–$R_{\rm hm}$ plane at $Z = 0.002$ showing, from top to bottom: the number of collisions over the total number of interactions, $N_{\rm coll}/N_{\rm int}$; the number of VMSs with mass $m_* \geq 300 $\Ms$$ over the total number of simulated binaries, $N_{\rm VMS}/N_{\rm tot}$; the mean maximum stellar mass. In all panels, the shaded regions indicate the typical ranges of $R_{\rm hm}$ and $M_{\rm cl}$ for clusters in the Cosmic Gems Arc (CGA), young clusters (YCs) from [Messa, et. al (2026)](https://ui.adsabs.harvard.edu/abs/2026A&A...705A.173M), and globular clusters (GCs), and nuclear star clusters (NSCs) from [Paiella, et. al (2026)](https://ui.adsabs.harvard.edu/abs/2026A&A...708A.200P). The dashed line marks clusters with a core-collapse timescale $t_{\rm cc} = 5$ Myr. All cluster properties are considered at the present known age. (*fig:contour_coll_vms_growth*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.03812"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\Nthresh}{\mbox{N_{\rm threshold}}}$
$\newcommand{\ee}[1]{\mbox{  \times 10^{#1}}}$
$\newcommand{\av}{\mbox{A_{\rm V}}}$
$\newcommand{\mdense}{M_{\mbox{\scriptsize sgb}}}$
$\newcommand{\brevise}[1]{{\footnotesize \color{magenta}[HBLrevised: #1]}}$
$\newcommand{\pin}{\mathcal{PI} }$
$\newcommand{\mle}{\mathcal{MLE} }$
$\newcommand{\mae}{\mathcal{MAE} }$
$\newcommand\purple{#1 }$
$\newcommand\red{#1 }$
$\newcommand\blue{#1 }$
$\newcommand\zy{#1 }$
$\newcommand\Htwo{ H_2 }$
$\newcommand\cmt{ cm^{-3} }$
$\newcommand\Msgb{ M_{\rm sgb} }$
$\newcommand\reference{#1 }$</div>



<div id="title">

# Constraining Dust Properties Along the Minor Axis of Andromeda

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.04319-b31b1b.svg)](https://arxiv.org/abs/2611.04319)<mark>Appeared on: 2026-11-04</mark> -  _15 pages, 12 figures. Accepted for publication in A&A_

</div>
<div id="authors">

<mark>S. Jiao</mark>, et al. -- incl., <mark>H. Beuther</mark>, <mark>K. Doi</mark>, <mark>J. d. Brok</mark>, <mark>F. Xu</mark>, <mark>S. D. Reyes-Reyes</mark>

</div>
<div id="abstract">

**Abstract:** Previous studies of dust emission in M31 based primarily on $_ Herschel_$ data have reported inconsistent radial trends in the dust emissivity index ( $\beta$ ), partly due to limited wavelength coverage and the intrinsic $T_{\rm dust}$ – $\beta$ degeneracy. We aim to improve the constraints of $\beta$ determinations and obtain the spatial surface density and temperature distributions at GMC scales in M31 by extending the spectral coverage into the millimeter regime and by quantifying the impact of wavelength coverage on the $T_{\rm dust}$ – $\beta$ degeneracy. We present new IRAM 30m NIKA2 1.15 mm observations along the minor axis of M31 at an angular resolution of $11"$ , corresponding to $\sim$ 40 pc at the distance of M31.These data are used with $_ Herschel_$ and JCMT SCUBA-2 maps to perform pixel-by-pixel single-component modified blackbody fitting. Including the 850 $\mu$ m and 1.15 mm data helps reduce the $T_{\rm dust}$ – $\beta$ degeneracy and yields an average emissivity index of $\beta \approx 1.8$ , systematically lower than values derived from far-infrared–only analyses.Radial analysis shows higher $\beta$ values in the 5 kpc ring compared to the 10 kpc ring.The elevated $\beta$ values in the inner 5 kpc may be linked to changes in dust composition, potentially associated with enhanced starlight heating in this region. Millimeter wavelength coverage is important for improving $\beta$ constraints in M31.The lower $\beta$ values inferred from the 250--1150 $\mu$ m SED fitting suggest possible variations in dust emissivity and highlight the importance of sampling the long-wavelength SED slope in nearby galaxies.

</div>

<div id="div_fig1">

<img src="tmp_2611.04319/./Figures/Figure3.png" alt="Fig10" width="100%"/>

**Figure 10. -** 
The distribution of dust surface density (18$"$ angular resolution), temperature (18$"$ angular resolution), and $\beta$(37$"$ angular resolution) obtained from the iterative SED fits using 250--1150 $\mu$m data.
The gray dashed line delineates the field of view of the new NIKA2 observations, while the black dashed contour corresponds to the 2 mJy beam$^{-1}$ sensitivity limit of the NIKA2 1.15 mm data.
The white contours trace the spiral-arm mask from [Jiao, et. al (2026)](https://ui.adsabs.harvard.edu/abs/2026ApJS..282...39J).
The cyan circles in the dust surface density map mark the three example positions used for the SED comparisons in Section \ref{subsec:T-beta}. (*fig:sed_maps*)

</div>
<div id="div_fig2">

<img src="tmp_2611.04319/./Figures/Figure4_radial_distribution_Sigma_T_beta.png" alt="Fig1" width="100%"/>

**Figure 1. -** 
Results from the 250--1150 $\mu$m band iterative SED fits for each pixel plotted vs. radius.
The blue points show the fitted value for each pixel, while the yellow squares represent averages in 700 pc bins.
The corresponding yellow error bars represent the standard deviation within each bin.
The dashed blue lines represent the fitted relationships from the HELGA II  ([Smith, et. al 2012](https://ui.adsabs.harvard.edu/abs/2012ApJ...756...40S)) .
The histograms on the right side show the dust surface density, dust temperature, and $\beta$ distribution of the map region shown in Figure \ref{fig:sed_maps}. (*fig:ori_sed_high_nh2*)

</div>
<div id="div_fig3">

<img src="tmp_2611.04319/./Figures/Figure6.png" alt="Fig2" width="100%"/>

**Figure 2. -** 
Joint posterior distributions of $T_{\rm dust}$ and $\beta$ derived from SED fitting with different band coverages.
The black and orange contours represent fits using the 250-1150 $\mu$m band and 70-1150 $\mu$m band respectively.
Solid and dashed contours correspond to the 0.5 and 0.9 density levels.
The case using 250-500 $\mu$m bands is shown as gray scatter points.
Marginal posterior distributions of $T_{\rm dust}$ and $\beta$ are shown in the top and right panels.
Colored dashed lines and square markers indicate the median values of each distribution. (*fig:T-beta*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.04319"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$</div>



<div id="title">

# The ALMA survey to Resolve exoKuiper belt Substructures (ARKS): XII. Constraints on large bodies embedded in discs

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.04976-b31b1b.svg)](https://arxiv.org/abs/2611.04976)<mark>Appeared on: 2026-11-04</mark> -  _accepted for publication in A&A_

</div>
<div id="authors">

M. R. Jankovic, et al. -- incl., <mark>T. Henning</mark>

</div>
<div id="abstract">

**Abstract:** The ALMA survey to Resolve exoKuiper belt Substructures (ARKS) presents detailed constraints on structure and properties of a sample of 24 debris discs at millimetre wavelengths, including measurements of vertical thicknesses for a subsample of discs. We estimated sizes and total masses of large bodies that feed collisional cascades producing the observed millimetre dust in 11 ARKS discs with a well constrained vertical thickness. Separately, we estimated the sizes and total masses of large bodies that could be gravitationally stirring these discs, assuming that such stirring is the origin of the vertical thickness. We used a semi-analytic particle-in-a-box collisional model and a semi-analytic model of stirring through gravitational scattering. We accounted for the dependence of collisional damping efficiency on the average collision velocity. We find that planetesimals feeding the collisional cascades are less than 20 km in size, while the total masses in the planetesimals are lower than 200 M $_\oplus$ . There is a good agreement between minimum disc masses inferred from millimetre and infrared observations. The hypothetical stirrers are always larger in size than the planetesimals, but they do not necessarily contain most of the disc mass. In agreement with previous work, the observed dust masses can be explained by steady-state collisional models with reasonable disc masses. The measured vertical scale-heights can be explained by gravitational stirring from embedded large bodies of reasonable properties. However, more advanced models are needed to address the observed non-Gaussian vertical profiles, along with the dependence of disc scale-height on particle size and stellocentric distance.

</div>

<div id="div_fig1">

<img src="tmp_2611.04976/./m_pl.png" alt="Fig3" width="100%"/>

**Figure 3. -** Planetesimal sizes feeding the cascades assuming primordial power-law size distribution with slope $q_{\rm p}=3$(circles) and its extrapolation up to 200 km (thin lines), and likewise for $q_{\rm p}=4$(squares and thick lines), plotted against the corresponding mass inside unit log bin of size around the planetesimal size. For a size distribution with slope $q_{\rm p}=4$ there is equal mass in each log bin of size. (*fig:m_pl*)

</div>
<div id="div_fig2">

<img src="tmp_2611.04976/./m_M_together.png" alt="Fig5" width="100%"/>

**Figure 5. -** Joint constraints on sizes of planetesimals feeding the collisional cascades (circles for $q_{\rm p}=3$ and squares for $q_{\rm p}=4$, the same as in Fig. \ref{fig:m_pl}) and on the stirrers exciting particles' inclinations (solid lines). The constraints on the stirrers show plausible combinations of their total mass and their individual size, and not a size distribution. These constraints account for collisional damping of mm-sized dust in discs in which it is estimated to be efficient. (*fig:together*)

</div>
<div id="div_fig3">

<img src="tmp_2611.04976/./crit_ratio.png" alt="Fig1" width="100%"/>

**Figure 1. -** Critical projectile-to-target mass ratio as function of target size for the discs in our sample, obtained using Eqs. (\ref{eq:v_rel}), (\ref{eq:crit_mass_ratio}), and (\ref{eq:q_d_star}), and using data provided in Table \ref{table:data}. The horizontal gray line indicates where $2 Q_{\rm D}^*/v_{\rm rel}^2 = 1$.  (*fig:crit_ratio*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.04976"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

133  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

7  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

5  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
